# Q-value threshold calibration (Phase III)

Tune legacy Python Q-value cuts against Malofeeva IR binaries.
Requires `m34_join.csv` and pipeline Q values (`midas_pipeline.csv`).

From repo root:
```bash
cd research && source .venv/bin/activate
python scripts/cross_match.py
python scripts/validate_phase3.py --refresh-pipeline --ebv 0.07
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "midas").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from midas.validation import (
    load_validation_rows,
    sweep_q_thresholds,
    validate_malofeeva,
    validate_roc_malofeeva,
)

EBV = 0.07
rows = load_validation_rows(ebv=EBV, refresh_pipeline=False)
print(f"Loaded {len(rows)} stars with Q values (E(B-V)={EBV})")

In [ ]:
grid = sweep_q_thresholds(rows, members_only=True)
df = pd.DataFrame(grid)
df.head(10)

In [ ]:
best = df.iloc[0]
print(
    f"Best F1={best.f1:.3f} at Q∈({best.q_low}, {best.q_high}] "
    f"(P={best.precision:.3f}, R={best.recall:.3f})"
)

baseline = validate_malofeeva(rows, q_low=0.0, q_high=1.0)
print(
    f"Baseline Q∈(0,1]: F1={baseline['f1']:.3f} "
    f"P={baseline['precision']:.3f} R={baseline['recall']:.3f}"
)

In [ ]:
roc = validate_roc_malofeeva(rows)
curve = pd.DataFrame(roc["curve"])

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(curve["fpr"], curve["tpr"], color="#e8c547", lw=2)
ax.plot([0, 1], [0, 1], "--", color="#666", lw=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate (Malofeeva)")
ax.set_title("ROC — Q score vs Malofeeva IR")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot(index="q_low", columns="q_high", values="f1")
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(pivot.values, origin="lower", cmap="viridis", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{x:.1f}" for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{x:.1f}" for x in pivot.index])
ax.set_xlabel("q_high")
ax.set_ylabel("q_low")
ax.set_title("F1 vs Malofeeva (CG members)")
plt.colorbar(im, ax=ax, label="F1")
plt.tight_layout()
plt.show()